# TRUSTDESC — complete available Colab experiment workflow

**Primary Part 2 entry point. Run sections 1–16 in order.** This notebook orchestrates the repository's actual modules. It does not certify a complete original-paper reproduction. Exact author resources were not recovered in the final bounded search. Included experiments are labeled synthetic; the public-source case is static-only; generated alternative tasks are reconstructed.

Start with **CPU** for the no-credential workflow, or select **T4 GPU** and explicitly add `qwen-small` in section 4 for alternative-model inference. High-memory GPUs can optionally run the separately selected 7B model. Paper model names are preserved; paid calls require credentials and typed confirmation for the exact model configuration. **No unknown MCP implementation executes on the Colab host.**

No Drive mount or Codex/ChatGPT account connection is needed. Use the final portable project ZIP plus checksum, or your private GitHub clone. Download checkpoints/results regularly; Colab storage is ephemeral. In the review section, inspect traces before advancing to final analysis/export. Running all cells unattended leaves success metrics unreviewed, rather than inventing labels.


## 1. Inspect the runtime before installing anything

In [ ]:
import sys, os, platform, shutil, subprocess, json, re
from pathlib import Path
assert sys.version_info >= (3, 11), "Use Python 3.11+"
ram = {}
if Path("/proc/meminfo").exists():
    ram = {k: int(v)*1024 for k,v in re.findall(r"^(MemTotal|MemAvailable):\s+(\d+)", Path("/proc/meminfo").read_text(), re.M)}
gpu_text = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else "No NVIDIA GPU / CUDA driver detected"
print(json.dumps({"python":sys.version,"RAM_bytes":ram,"disk_free_bytes":shutil.disk_usage(".").free,"runtime_type":"Colab" if Path("/content").exists() else "local validation"}, indent=2))
print(gpu_text)


## 2. Set up the verified project

Default `SETUP_MODE="upload"`: upload **only the final project ZIP**, copy the SHA-256 from its companion file, and run. Result ZIPs belong to section 3, not this cell. This verifies member hashes before importing project code.

Alternative `SETUP_MODE="clone"`: the repository is private. Supply a read-only token through Colab Secret `TRUSTDESC_GITHUB_TOKEN` or hidden input. It is used in an ephemeral subprocess environment, never stored in Git configuration, URLs, experiment files or exported results. The project remote remains a normal HTTPS URL. No Google Drive mount is performed.


In [ ]:
SETUP_MODE = "upload"  # "upload" or "clone"
EXPECTED_PROJECT_SHA256 = ""  # From the supplied .sha256 file; hidden/ordinary input below if blank.
from google.colab import files
import hashlib, zipfile, base64, getpass
from pathlib import PurePosixPath
ROOT = Path("/content/trustdesc-reproduction")
if ROOT.exists():
    assert (ROOT / "src/trustdesc/experiments/workflow.py").is_file(), "Existing folder is not the prepared project"
    print("Using existing extraction/clone; its code hash will be bound to this run.")
elif SETUP_MODE == "upload":
    uploaded = files.upload()
    assert len(uploaded) == 1
    filename, data = next(iter(uploaded.items()))
    expected = EXPECTED_PROJECT_SHA256 or input("Paste project ZIP SHA-256: ").strip()
    assert hashlib.sha256(data).hexdigest() == expected, "Project archive checksum mismatch"
    import io
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        infos = z.infolist()
        assert len(infos) <= 20000 and sum(i.file_size for i in infos) < 128_000_000
        assert len(z.namelist()) == len(set(z.namelist()))
        for i in infos:
            p = PurePosixPath(i.filename)
            assert not p.is_absolute() and ".." not in p.parts and "\\" not in i.filename
            assert p.parts[0] == "trustdesc-reproduction" and ((i.external_attr >> 16) & 0o170000) != 0o120000
        manifest = json.loads(z.read("trustdesc-reproduction/PORTABLE_MANIFEST.json"))
        assert set(z.namelist()) == {"trustdesc-reproduction/"+n for n in manifest["files"]} | {"trustdesc-reproduction/PORTABLE_MANIFEST.json"}
        for n, sha in manifest["files"].items():
            assert hashlib.sha256(z.read("trustdesc-reproduction/"+n)).hexdigest() == sha
        z.extractall("/content")
    print({"verified_project_files":len(manifest["files"]),"source_commit":manifest["git_commit"]})
elif SETUP_MODE == "clone":
    def private_clone():
        try:
            from google.colab import userdata
            token = userdata.get("TRUSTDESC_GITHUB_TOKEN")
        except Exception:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        if not token: raise RuntimeError("No GitHub access supplied; use verified ZIP upload")
        env = {k:v for k,v in os.environ.items() if not k.startswith(("GIT_TRACE","GIT_CURL_VERBOSE"))}
        env.update(GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0="AUTHORIZATION: basic "+base64.b64encode(("x-access-token:"+token).encode()).decode())
        try:
            p = subprocess.run(["git","-c","credential.helper=","clone","--branch","main","https://github.com/SamarthRamesha/trustdesc-reproduction.git",str(ROOT)],env=env,capture_output=True,text=True)
            if p.returncode: raise RuntimeError("Private clone failed; use verified project upload or correct repository access")
        finally:
            env.clear(); token = None
    private_clone()
    print({"cloned_commit":subprocess.run(["git","rev-parse","HEAD"],cwd=ROOT,capture_output=True,text=True,check=True).stdout.strip()})
else: raise ValueError("Unknown setup mode")


In [ ]:
ROOT = Path(ROOT).resolve()
assert (ROOT / "src/trustdesc/experiments/workflow.py").is_file()
sys.path.insert(0, str(ROOT / "src"))
from trustdesc.experiments.runtime import inspect_runtime
from trustdesc.experiments.setup import prepare_inputs, extract_resource_bundle
from trustdesc.experiments.workflow import Workflow
from trustdesc.experiments.models import cost_preview
from trustdesc.experiments.review import review_widget, import_reviews
from trustdesc.experiments.report import analyze
from trustdesc.experiments.export import export_results, restore_results
from trustdesc.common.io import digest
ENV = {**os.environ, "PYTHONPATH":str(ROOT / "src")}
RUNTIME = inspect_runtime()
print(json.dumps(RUNTIME, indent=2))
print("Recommended capacity profile:", RUNTIME["recommended_profile"], "— no model has been changed or executed.")


## 3. Restore an earlier result checkpoint (optional)

Leave `RESTORE_RESULTS=False` for a new run. After a disconnect, upload your own `TRUSTDESC_COLAB_RESULTS.zip` and provide its checksum. The function verifies every member and restores to a fresh folder with the original run ID. It imports no code from the result archive. Recreate the same project/data/model configuration; mismatches will refuse resume. Re-enter credentials if needed—none are restored. In-flight requests may already have incurred cost; explicitly resuming may repeat their incomplete stage.


In [ ]:
RESTORE_RESULTS = False
RESTORED_RUN = globals().get("RESTORED_RUN", None)
if RESTORE_RESULTS:
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1
    filename, data = next(iter(uploaded.items()))
    archive_path = Path("/content/restore-results.zip"); archive_path.write_bytes(data)
    expected = input("Paste this RESULT archive SHA-256: ").strip()
    RESTORED_RUN = restore_results(archive_path, expected, Path("/content/restored-runs"))
    print("Verified restored run:", RESTORED_RUN)
else: print("No checkpoint archive requested.")


## 4. Central experiment/model settings — edit only this cell

Profiles: **A** CPU/replay/static; **B** standard GPU/small alternative; **C** high-memory/explicitly selected larger alternative; **D** configured paper model endpoints. `auto` recommends capacity only. Model choice is always explicit. Default `SELECTED_MODELS=["replay"]` makes no inference calls and needs no credentials.

For a T4, explicitly add `qwen-small` and set `ALLOW_MODEL_DOWNLOADS=True`. For sufficient high-memory GPU, `qwen-large` is a different 7B alternative. No automatic downgrade or original-model substitution occurs. The Qwen weights/revisions are fixed public resources, not original paper models. If capacity/provider/dependencies are unavailable, that configuration records failure/unavailability and other configurations continue.

Original paper configurations are `paper-claude`, `paper-gemini`, `paper-gpt`, `paper-oss`. All preserve temperature .2 and provider-default reasoning. Set their **verified provider model IDs** and endpoints in `MODEL_OVERRIDES`; the paper supplies display names but no exact API IDs/snapshots. `paper-oss` needs an explicitly configured remote endpoint/infrastructure: four RTX PRO 6000 Blackwell GPUs were used in the paper; a normal Colab GPU is not equivalent. Missing settings produce explicit unavailable records, not fake original results.

Use `BENCHMARK_INPUT="available"` for the included eight synthetic tasks plus optional public static source. A genuine resource bundle can be uploaded in section 7 with `BENCHMARK_INPUT="supplied"`; it is never silently replaced. No original 52 tools/208 tasks are manufactured.


In [ ]:
PROFILE = "auto"
SELECTED_MODELS = ["replay"]  # e.g. ["replay", "qwen-small"] for an explicitly selected T4 alternative experiment
MODEL_OVERRIDES = {}  # e.g. {"paper-gemini":{"model_id":"verified-provider-id","price_per_million":{"input":None,"output":None}}}
ALLOW_MODEL_DOWNLOADS = False
INSTALL_CPU_DEPENDENCIES = True
BENCHMARK_INPUT = "available"  # "available" or "supplied"
ACQUIRE_PUBLIC_SOURCE = True  # pinned paper-linked arxiv source, parsed only; failure does not stop fixtures
UPLOAD_RESOURCE_BUNDLE = False
RESOURCE_BUNDLE_PATH = ""
GENERATE_RECONSTRUCTED_TASKS = False  # real model + public source only; records two original-metadata-derived/two generated-description-derived tasks
RESUME_DIRECTORY = str(RESTORED_RUN) if RESTORED_RUN else ""
RESUME_CHOICE = "ask"  # Explicitly type RESUME below for an existing run; or set "resume"
RETRY_FAILED = False  # True explicitly retries failed stages; may issue new model calls
AGENT_MAX_STEPS = 6
DOWNLOAD_RESULTS = True

# Local validation overrides only; the shipped notebook has no machine-specific paths.
VALIDATION_OPTIONS = globals().get("VALIDATION_OPTIONS", {})
ACQUIRE_PUBLIC_SOURCE = VALIDATION_OPTIONS.get("acquire_public", ACQUIRE_PUBLIC_SOURCE)
DOWNLOAD_RESULTS = VALIDATION_OPTIONS.get("download_results", DOWNLOAD_RESULTS)
INSTALL_CPU_DEPENDENCIES = VALIDATION_OPTIONS.get("install_dependencies", INSTALL_CPU_DEPENDENCIES)
registry = json.loads((ROOT / "configs/colab_models.json").read_text())["models"]
import copy
MODELS = []
for name in SELECTED_MODELS:
    model = copy.deepcopy(next(m for m in registry if m["id"] == name))
    model.update(MODEL_OVERRIDES.get(name, {})); MODELS.append(model)
CONFIG = {"profile":RUNTIME["recommended_profile"] if PROFILE == "auto" else PROFILE,
          "models":MODELS,"agent_max_steps":AGENT_MAX_STEPS,"generate_reconstructed_tasks":GENERATE_RECONSTRUCTED_TASKS}
print(json.dumps(CONFIG, indent=2))
print("Model choice is explicit. No paid model is authorized by selecting a name alone.")


## 5. Install only the selected dependencies and validate software

CPU pipeline uses the standard library; plots/review UI use the CPU requirements. GPU packages install only after explicit download opt-in and inspected GPU capacity. A Colab package/runtime restart may be required by the package manager; rerun imports/configuration afterwards without overwriting source or result folders. Proposed GPU dependency compatibility is not locally certified. No weights are downloaded until a selected model is first used.


In [ ]:
DEPENDENCY_LOG = []
if INSTALL_CPU_DEPENDENCIES:
    p = subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-r",str(ROOT / "notebooks/colab/requirements-cpu.txt")],capture_output=True,text=True)
    DEPENDENCY_LOG.append({"track":"cpu","exit_code":p.returncode,"output":p.stdout+p.stderr})
    if p.returncode: print("CPU plot/UI dependency install failed; core experiments still run, plot/UI status will be explicit.")
if ALLOW_MODEL_DOWNLOADS and RUNTIME["gpus"] and any(m["provider"] == "transformers" and RUNTIME["gpus"][0]["free_bytes"] >= m["minimum_gpu_bytes"] for m in MODELS):
    p = subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-r",str(ROOT / "notebooks/colab/requirements-gpu.txt")],capture_output=True,text=True)
    DEPENDENCY_LOG.append({"track":"gpu","exit_code":p.returncode,"output":p.stdout+p.stderr})
    if p.returncode: print("GPU dependency failure preserved; other configured experiments continue.")
RUNTIME = inspect_runtime()  # actual post-install versions are part of resume binding
print(json.dumps({"dependency_status":[{k:r[k] for k in ["track","exit_code"]} for r in DEPENDENCY_LOG],"runtime":RUNTIME}, indent=2))


In [ ]:
p = subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=ROOT,env=ENV,capture_output=True,text=True)
TEST_LOG = p.stdout+p.stderr
print(TEST_LOG[-2000:])
assert p.returncode == 0, "Core tests failed; preserve output and inspect before experiments"


## 6. Acquire/validate data and prepare source as text

Default: eight authored synthetic tasks, two fixture tools; optional pinned Apache-2.0 current arxiv server for **static inspection only**. Source archive failure is recorded and does not stop CPU fixtures. It is not the original 52-tool dataset. No downloaded README/agent instructions/setup/tests/server code execute.

For recovered resources, set `BENCHMARK_INPUT="supplied"` and upload a checksum-verified ZIP containing exactly one `bundle.json` plus source text. The contract is `configs/original_benchmark_contract.json` and the guide describes all required fields. Author-candidate bundles require 52 tools and 208 actual supplied tasks; provenance must be independently checked. External backends remain disabled even when author resources exist. Original success labels cannot silently label newly produced traces.


In [ ]:
from datetime import datetime, timezone
import uuid
WORKSPACE = ROOT / "data/interim" / ("colab-inputs-" + uuid.uuid4().hex)
if UPLOAD_RESOURCE_BUNDLE:
    from google.colab import files
    uploaded = files.upload(); assert len(uploaded) == 1
    name, data = next(iter(uploaded.items()))
    p = Path("/content/resource-bundle.zip"); p.write_bytes(data)
    RESOURCE_BUNDLE_PATH = str(extract_resource_bundle(p, input("Resource archive SHA-256: ").strip(), WORKSPACE / "supplied"))
if BENCHMARK_INPUT == "supplied" and not RESOURCE_BUNDLE_PATH:
    raise ValueError("Supply genuine/reconstructed bundle.json; no synthetic replacement is made")
BUNDLE, SOURCE_ROOTS, ACQUISITION = prepare_inputs(ROOT, WORKSPACE, acquire_public=ACQUIRE_PUBLIC_SOURCE,
    resource_bundle=RESOURCE_BUNDLE_PATH if BENCHMARK_INPUT == "supplied" else None)
print(json.dumps({"origin":BUNDLE["origin"],"tools":len(BUNDLE["tools"]),"tasks":len(BUNDLE["tasks"]),"acquisition":ACQUISITION}, indent=2))


## 7. Credential and paid execution confirmation

This cell does nothing for replay/public local models. For each HTTP configuration it displays the selected model/endpoint, source/description/task/trace transmission scope, request cap and a cost estimate **only when verified rates are supplied**. Unknown prices remain unknown; the planning estimate is not a billing cap. Declining confirmation records unavailable configurations and allows others to continue. The paper's published historical cost is not a current price quote.

Colab Secret `TRUSTDESC_API_KEY` or hidden input is held in memory only. Do not enter keys into CONFIG/overrides, review fields or task files. Request caps persist across resume; incomplete in-flight requests may already be billed.


In [ ]:
CREDENTIALS = {}; APPROVED_HTTP = set()
for model in MODELS:
    if model["provider"] != "http": continue
    preview = cost_preview(model)
    preview["selected_data"] = {"benchmark_origin":BUNDLE["origin"],"tasks":len(BUNDLE["tasks"]),"tools":[{"id":t["id"],"origin":t["origin"],"backend":t["backend"]} for t in BUNDLE["tools"]]}
    print(json.dumps(preview, indent=2))
    if not model.get("model_id") or not model.get("endpoint"):
        print("Unavailable: supply the verified model ID/endpoint in the central settings cell; no credentials requested."); continue
    required = "RUN " + model["id"] + " " + digest(model)[:8]
    answer = input("To authorize this exact remote/potentially paid configuration, type '"+required+"' (blank skips): ")
    if answer != required:
        print("Not authorized; this configuration will be recorded unavailable."); continue
    import getpass
    secret_name = model.get("credential_name", "TRUSTDESC_API_KEY")
    try:
        from google.colab import userdata
        key = userdata.get(secret_name)
    except Exception:
        key = getpass.getpass("Provider API key (not displayed/stored): ")
    if key:
        CREDENTIALS[secret_name] = key; APPROVED_HTTP.add(digest(model))
    key = None
print({"approved_configuration_count":len(APPROVED_HTTP),"credentials_printed":False})


## 8. Create a unique run or explicitly resume

Completed tool/stage/task checkpoints are reused only when code, data, source, model/config, prompts, Python/packages and hardware identity match. Completed checkpoint hashes are checked. Incomplete attempts are not successful; failures are retained unless `RETRY_FAILED=True`. A changed configuration requires a new run. All model calls have durable pre-call records; a disconnect may leave an in-flight request whose cost/result is unknown.


In [ ]:
if not RESUME_DIRECTORY:
    candidates = sorted((ROOT / "experiments/runs").rglob("EXPERIMENT_MANIFEST.json")) if (ROOT / "experiments/runs").exists() else []
    if candidates:
        print({str(i):str(p.parent) for i,p in enumerate(candidates)})
        choice = input("Existing experiment runs detected. Enter an index to resume, or NEW: ").strip()
        if choice != "NEW": RESUME_DIRECTORY = str(candidates[int(choice)].parent)
if RESUME_DIRECTORY:
    RUN_ROOT = Path(RESUME_DIRECTORY)
    old = json.loads((RUN_ROOT / "EXPERIMENT_MANIFEST.json").read_text())
    print({"existing_run":old["run_id"],"status":old["status"],"fingerprint":old["fingerprint"]})
    if RESUME_CHOICE != "resume":
        assert input("Type RESUME to reuse compatible completed work (in-flight stages may repeat calls): ") == "RESUME"
    RESUME = True
else:
    RUN_ROOT = Path(VALIDATION_OPTIONS.get("run_directory", ROOT / "experiments/runs" / (datetime.now(timezone.utc).strftime("colab-%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8])))
    RESUME = False
W = Workflow(ROOT, RUN_ROOT, CONFIG, BUNDLE, SOURCE_ROOTS, RUNTIME, resume=RESUME, retry_failed=RETRY_FAILED,
             credentials=CREDENTIALS, approved_http=APPROVED_HTTP, allow_download=ALLOW_MODEL_DOWNLOADS)
W.store.write("ACQUISITION_REPORT.json", ACQUISITION)
W.store.write("authorization_record.json", {"approved_http_configuration_hashes":sorted(APPROVED_HTTP),"public_download_opt_in":ALLOW_MODEL_DOWNLOADS,"credentials_exported":False})
W.store.write("model_registry.json", json.loads((ROOT / "configs/colab_models.json").read_text()))
W.store.write("DEPENDENCY_REPORT.json", DEPENDENCY_LOG)
W.store.write("SOFTWARE_VALIDATION.json", {"exit_code":0,"command":"python -m unittest discover -s tests -v","output":TEST_LOG,"actual_colab_session":RUNTIME["runtime_type"] == "colab"})
W.store.text("published_reference.csv", (ROOT / "reports/published_results.csv").read_text())
print({"run":str(RUN_ROOT),"fingerprint":W.fingerprint,"resumed":RESUME,"external_execution":"disabled"})


## 9. SliceMin — actual static analysis and per-model optional debloating

In [ ]:
STATIC = W.static()
print({tool:{"status":r["status"],"reachable_functions":len(r.get("slice",{}).get("functions",[])),"unresolved_calls":len(r.get("slice",{}).get("unresolved_calls",[]))} for tool,r in STATIC.items()})
print("Slices, call sites, source hashes and warnings are checkpointed. Unvalidated TypeScript is unavailable, not silently approximated.")


## 10. DescGen — actual repository sanitization/generation

For configured models this performs real inference. Replay is labeled software validation. To request per-function model debloating, set `debloat=True` for that selected real model in MODEL_OVERRIDES before starting a **new run**. Raw safe model output, request/prompt hashes, sanitized source, descriptions/schema/claims, validation failures and partial results are persisted. A malformed small-model response remains a failure; no silent parsing repair or model substitution.


In [ ]:
DRAFTS = W.descriptions()
print({model:{tool:r["status"] for tool,r in values.items()} for model,values in DRAFTS.items()})


## 11. DynVer — planned tasks, actual controlled calls, judgment and refinement

The existing verifier executes only the fixed authored allowlist. External tools use DisabledRunner; statements without observations become inconclusive. Controlled fixture execution is not original LangChain/MCP verification. Semantic planner/judge responses are replayed only in replay mode; configured live models use actual requests. Supported/refuted/inconclusive decisions, task logs, full/lite descriptions and schema limitations are saved.


In [ ]:
VERIFICATION = W.verification()
print({model:{tool:{"status":r["status"],"backend":r.get("execution_backend"),"usable_summary":r.get("verification",{}).get("usable_summary")} for tool,r in values.items()} for model,values in VERIFICATION.items()})


## 12. Original / lite / full benchmark conditions

The runner passes each condition's actual descriptions to the controlled agent, tracks every task/model/condition, and saves target-call observations, latency/tokens/actual cost when reported, raw calls and errors. Eight default tasks × three conditions = 24 records per model. Model actions/answers are **not success labels**.

Optional real-model public-source task reconstruction produces two tasks from current original metadata and two from the generated description, labels them reconstructed and preserves generation provenance. They are not the author's 208 tasks and cannot execute without an isolated backend. Every selected configuration continues independently after another fails.


In [ ]:
PREDICTIONS = W.benchmark()
from collections import Counter
print({"task_records":len(PREDICTIONS),"statuses":dict(Counter(r["status"] for r in PREDICTIONS)),"objective_labels":"unreviewed until your explicit review","run":str(RUN_ROOT)})


## 13. Human review — inspect before advancing

Choose each executed record, inspect the task and actual trace, then explicitly label **target invoked** and **objective achieved**. Both are required for success. An error is not automatically failure or success: a task asking to try an invalid formula may require the observed rejection. Never copy a model's final assertion as ground truth. Complete the available controlled tasks for all conditions to obtain comparable TSR and common-success overhead.

You may leave records unreviewed and export partial evidence; those metrics remain blank. Missing external execution cannot be reviewed into a success. Click Save for each judgment, then continue to section 14. Alternatively download/edit `REVIEW_TEMPLATE.csv` with true/false labels, reviewer and unchanged result/evidence hashes, and import it in the next cell.


In [ ]:
if VALIDATION_OPTIONS.get("skip_review_ui", False):
    print("Local validation: interactive human review not executed; success metrics must remain null.")
else:
    try: review_widget(RUN_ROOT)
    except ImportError: print("Review UI unavailable: use REVIEW_TEMPLATE.csv and the import cell; do not invent labels.")


In [ ]:
UPLOAD_REVIEW_CSV = False
if UPLOAD_REVIEW_CSV:
    from google.colab import files
    uploaded = files.upload(); assert len(uploaded) == 1
    name, data = next(iter(uploaded.items()))
    path = Path("/content/operator-reviews.csv"); path.write_bytes(data)
    print({"human_reviews_imported":import_reviews(RUN_ROOT, path)})
else: print("No review CSV requested. Widget reviews are already persisted.")


## 14. Measured tables, metrics, plots and report

TSR is calculated only after all eligible condition tasks have explicit review. Blocked external/unconfigured data are reported separately. Runtime/cost overhead uses tasks successful in all three conditions. Unknown costs/latencies/labels stay missing, never zero-filled. Replay's measured API billing is zero because it performs no inference. Published paper values remain separate references; these fixture/reconstructed results are not numerically comparable with the paper.


In [ ]:
RESULTS = analyze(RUN_ROOT)
print(json.dumps({"dataset":RESULTS["dataset_inventory"],"conditions":RESULTS["conditions"],"overheads":RESULTS["overheads"],"failure_breakdown":RESULTS["failure_breakdown"]}, indent=2))
print((RUN_ROOT / "COLAB_EXPERIMENT_REPORT.md").read_text())


In [ ]:
PLOTS = json.loads((RUN_ROOT / "plot_inventory.json").read_text())
if not VALIDATION_OPTIONS.get("skip_plot_display", False):
    from IPython.display import display, Image
    for item in PLOTS:
        if item.get("path"): display(Image(filename=str(RUN_ROOT / item["path"])))
print(json.dumps(PLOTS, indent=2))


## 15. One-click complete result export for Part 3

Run after review, or whenever you need a disconnect checkpoint. Analysis is regenerated from the **current** explicit reviews before export. Only registered experiment outputs are included: configurations/provenance/hashes, stage/checkpoint results, safe raw calls, traces, task/tool tables, review evidence, metrics, plots, errors and report. API keys, credentials, model caches, unrelated runtime/personal files are excluded; high-signal credential checks refuse suspicious exports.

Download **both** `TRUSTDESC_COLAB_RESULTS.zip` and `.sha256`. These are sufficient for Codex to finalize analysis without access to your Google account/session. Preserve failed/unreviewed runs; they are valid partial evidence, not successful reproduction. Export after substantial inference because local Colab storage can disappear.


In [ ]:
RESULTS = analyze(RUN_ROOT)
EXPORT = export_results(RUN_ROOT, RUN_ROOT.parent / "TRUSTDESC_COLAB_RESULTS.zip")
print(json.dumps(EXPORT, indent=2))
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(EXPORT["archive"])
    files.download(EXPORT["checksum_file"])


## 16. Stop preparation; continue in Colab / Part 3

Keep the project package/version, result ZIP and checksum. On restart use the same notebook/project, restore your own result archive in section 3, reselect the exact settings and source inputs, confirm resume and any paid calls, then execute remaining sections. Completed matching checkpoints make no new model requests. Changes to data/model/source/code/runtime identity require a new run; do not manually relabel stale outputs.

No further MacBook preparation prompt is needed. Part 3 receives the exported result ZIP/checksum and the repository URL. Actual GPU inference, paid-provider availability, original author resources and isolated arbitrary MCP execution are external conditions; they were not validated merely by compiling this notebook. The complete executable available workflow and its explicit limitations are documented in `docs/COLAB_EXPERIMENT_GUIDE.md`.
